In [0]:
from config.config import *

# Get access key securely from Key Vault via secret scope
sas_token = dbutils.secrets.get(SAS_SECRET_SCOPE, SAS_SECRET_KEY)


# IMPORTANT: Remove any key-based auth if present
spark.conf.unset(
    f"fs.azure.account.key.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net"
)

# Set SAS authentication
spark.conf.set(
    f"fs.azure.account.auth.type.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    "SAS"
)

spark.conf.set(
    f"fs.azure.sas.token.provider.type.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    "org.apache.hadoop.fs.azurebfs.sas.FixedSASTokenProvider"
)

spark.conf.set(
    f"fs.azure.sas.fixed.token.{STORAGE_ACCOUNT_NAME}.dfs.core.windows.net",
    sas_token
)


df_customers = spark.read.option("header", "true").csv(f"{BRONZE_PATH}customers.csv")
display(df_customers)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, when, to_date


In [0]:
def clean_customers(df):
    return (
        df
        .withColumn(
            "name",
            when(col("name").isNull(), "Unknown").otherwise(col("name"))
        )
        .withColumn(
            "city",
            when(col("city").isNull(), "Not Provided").otherwise(col("city"))
        )
        .filter(col("email").isNotNull())
        .withColumn("signup_date", to_date(col("signup_date"), "yyyy-MM-dd"))
        .withColumn("customer_id", col("customer_id").cast("int"))
    )


In [0]:
df_customers = clean_customers(df_customers)
display(df_customers)

In [0]:
df_orders = spark.read.option("header", "true").csv(f"{BRONZE_PATH}orders.csv")
display(df_orders)

In [0]:
def clean_orders(df):
    return (
        df
        .withColumn(
            "quantity",
            when(col("quantity").isNull(), 1).otherwise(col("quantity"))
        )
        .withColumn("quantity", col("quantity").cast("int"))
        .withColumn("order_date", to_date(col("order_date"), "yyyy-MM-dd"))
        .withColumn("order_id", col("order_id").cast("int"))
        .withColumn("customer_id", col("customer_id").cast("int"))
        .withColumn("product_id", col("product_id").cast("int"))
    )



In [0]:
df_orders = clean_orders(df_orders)
display(df_orders)

In [0]:
df_products = spark.read.option("header", "true").csv(f"{BRONZE_PATH}products.csv")
display(df_products)

In [0]:
def clean_products(df):
    return (
        df
        .withColumn(
            "price",
            when(col("price").isNull(), 0).otherwise(col("price"))
        )
        .withColumn("price", col("price").cast("int"))
        .withColumn("product_id", col("product_id").cast("int"))
    )


In [0]:
df_products = clean_products(df_products)
display(df_products)

In [0]:
def write_single_file(df, target_path, file_name):
    temp_path = f"{target_path}/_tmp_{file_name}"

    # 1. Write to temp folder as single partition
    (
        df.coalesce(1)
        .write
        .mode("overwrite")
        .option("header", True)
        .csv(temp_path)
    )

    # 2. Find the part file
    files = dbutils.fs.ls(temp_path)
    part_file = [f.path for f in files if f.name.startswith("part-")][0]

    # 3. Move part file to final location with desired name
    final_file_path = f"{target_path}/{file_name}.csv"
    dbutils.fs.mv(part_file, final_file_path)

    # 4. Delete temp folder (_SUCCESS, crc, etc.)
    dbutils.fs.rm(temp_path, recurse=True)



In [0]:
write_single_file(df_customers, SILVER_PATH, "clean_customers")
write_single_file(df_orders, SILVER_PATH, "clean_orders")
write_single_file(df_products, SILVER_PATH, "clean_products")
